# Implementare un sistema RAG (Retrieval-Augmented Generation)

In questo notebook impareremo a costruire un sistema **RAG** da zero. 

## Cos'è la RAG?
I modelli di linguaggio (LLM) come GPT-4 sono potenti ma hanno due limiti:
1.  **Conoscenza ferma al training**: Non sanno cosa è successo dopo il loro addestramento.
2.  **Dati privati**: Non conoscono i tuoi documenti aziendali o personali.

La RAG risolve questo problema fornendo al modello i dati giusti *al momento della domanda*.

## I Passaggi
1.  **Ingestion & Parsing**: Caricare e leggere i documenti.
2.  **Chunking**: Dividere il testo in pezzi piccoli.
3.  **Embedding**: Trasformare il testo in vettori numerici.
4.  **Indexing**: Salvare i vettori in un database (Vector Store).
5.  **Retrieval**: Cercare i pezzi rilevanti per la domanda.
6.  **Generation**: Usare i pezzi trovati per rispondere con l'LLM.

## 1. Setup e Configurazione

Importiamo le librerie necessarie e carichiamo le chiavi API.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    print("ATTENZIONE: OPENAI_API_KEY non trovata nel file .env")

## 2. Ingestion & Parsing (Lettura Dati)

Immaginiamo di voler interrogare un documento tecnico sulla nuova **Ducati Panigale V4 2025**. Abbiamo creato un file di testo in `data/ducati_specs.txt`.

Usiamo un `TextParser` per leggere il contenuto.

In [3]:
from datapizza.modules.parsers.docling import DoclingParser

# Percorso del file
file_path = "../data/ducati_specs.txt"

# Leggiamo il file (il parser potrebbe restituire una lista di documenti/nodi)
parser = DoclingParser()
documents = parser.parse(file_path)

print(f"Documenti caricati: {len(documents)}")
print(f"Anteprima contenuto:\n{documents[0].content[:200]}...")

/home/mcalcaterra/Documenti/GitHub/Datapizza/Ducati/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-29 03:11:25,518 - INFO - detected formats: [<InputFormat.XML_USPTO: 'xml_uspto'>]
2025-11-29 03:11:25,518 - ERROR - Input document ducati_specs.txt with format None does not match any allowed format: (dict_keys([<InputFormat.DOCX: 'docx'>, <InputFormat.PPTX: 'pptx'>, <InputFormat.HTML: 'html'>, <InputFormat.IMAGE: 'image'>, <InputFormat.PDF: 'pdf'>, <InputFormat.ASCIIDOC: 'asciidoc'>, <InputFormat.MD: 'md'>, <InputFormat.CSV: 'csv'>, <InputFormat.XLSX: 'xlsx'>, <InputFormat.XML_USPTO: 'xml_uspto'>, <InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.METS_GBS: 'mets_gbs'>, <InputFormat.JSON_DOCLING: 'json_docling'>, <InputFormat.AUDIO: 'audio'>, <InputFormat.VTT: 'vtt'>]))
2025-11-29 03:11:25,51

ConversionError: File format not allowed: ducati_specs.txt

## 3. Splitting (Chunking)

Non possiamo passare un intero libro all'LLM. Dobbiamo dividerlo in "pezzetti" (**chunks**) gestibili.

Usiamo il `RecursiveSplitter` che cerca di tagliare il testo rispettando paragrafi e frasi.

*   **chunk_size**: Grandezza massima del pezzo (in caratteri o token).
*   **chunk_overlap**: Sovrapposizione tra i pezzi per non perdere il contesto al taglio.

In [ ]:
from datapizza.splitters import RecursiveSplitter

splitter = RecursiveSplitter(
    chunk_size=300,    # Pezzi piccoli per questo esempio
    chunk_overlap=50   # 50 caratteri di sovrapposizione
)

# Splittiamo i documenti in 'nodi' più piccoli
nodes = splitter.split(documents)

print(f"Abbiamo ottenuto {len(nodes)} chunks dai documenti originali.")
print("--- Esempio Chunk 1 ---")
print(nodes[0].content)
print("--- Esempio Chunk 2 ---")
print(nodes[1].content)

## 4. Embedding & Vector Store (Indicizzazione)

Ora trasformiamo il testo in numeri (**vettori**). I vettori simili stanno vicini nello spazio matematico.

1.  **Embedder**: `OpenAIEmbedder` (usa modelli come `text-embedding-3-small`).
2.  **VectorStore**: `QdrantVectorStore` (un database veloce per vettori). Qui useremo la modalità "in-memory" (:memory:) per semplicità.

In [ ]:
from datapizza.embedders.openai import OpenAIEmbedder
from datapizza.vectorstores.qdrant import QdrantVectorStore

# 1. Inizializziamo l'Embedder
embedder = OpenAIEmbedder(api_key=os.getenv("OPENAI_API_KEY"))

# 2. Inizializziamo il Vector Store (Qdrant in memoria)
vector_store = QdrantVectorStore(
    collection_name="ducati_demo",
    location=":memory:"  # Database temporaneo in RAM
)

# 3. Creiamo l'indice
# Questa operazione calcola gli embedding per tutti i nodi e li salva in Qdrant
vector_store.add(nodes, embedder=embedder)

print("Indicizzazione completata!")

## 5. Retrieval (Ricerca)

Ora possiamo fare una domanda. Il sistema:
1.  Trasformerà la domanda in vettore.
2.  Cercherà nel Vector Store i vettori più vicini (Cosine Similarity).
3.  Restituirà i chunks di testo originali.

In [ ]:
query = "Quali sono le novità del telaio della Panigale V4?"

# Eseguiamo la ricerca (Retrieve)
# k=2 significa "dammi i 2 pezzi più rilevanti"
retrieved_nodes = vector_store.query(query, k=2, embedder=embedder)

print(f"Domanda: {query}\n")
for i, node in enumerate(retrieved_nodes):
    print(f"[Risultato {i+1}] (Score: {node.score:.4f}):")
    print(f"...{node.content}...")
    print("-" * 40)

## 6. Generation (Risposta Finale)

Infine, uniamo tutto. Creiamo un prompt che include:
1.  La domanda dell'utente.
2.  Il contesto recuperato (i chunk trovati).
3.  L'istruzione "Rispondi usando solo il contesto fornito".

In [ ]:
from datapizza.clients.openai import OpenAIClient

client = OpenAIClient(api_key=os.getenv("OPENAI_API_KEY"), temperature=0)

# Costruiamo il contesto unendo i testi recuperati
context_text = "\n\n".join([node.content for node in retrieved_nodes])

# Creiamo il prompt aumentato
augmented_prompt = f"""
Sei un assistente esperto Ducati. Usa LE SEGUENTI INFORMAZIONI per rispondere alla domanda dell'utente.
Se non trovi la risposta nel testo, dì che non lo sai.

CONTESTO:
{context_text}

DOMANDA UTENTE:
{query}
"""

print("--- Generazione della risposta ---")
response = client.invoke(augmented_prompt)
print(response.text)

## Conclusione

Hai appena costruito una pipeline RAG completa! 

### Riepilogo flusso:
`Dati -> Parser -> Splitter -> Embedder -> VectorStore -> (Query) -> Retriever -> LLM -> Risposta`

In produzione, `datapizza-ai` offre astrazioni come `IngestionPipeline` per automatizzare questi passaggi, ma capire i singoli componenti è fondamentale per il debugging e l'ottimizzazione.